In [2]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Importing the dataset
df = pd.read_excel(r"C:\Users\NEW USER\Downloads\DSN_Bootcamp\DSN_Bootcamp Analytics Dataset.xlsx")

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip()
)
print(df.head())

   transaction_id       date store_id  state   channel product_category  \
0          100001 2026-05-25    AB-02  Abuja  WhatsApp             Home   
1          100002 2026-04-17    LG-01  Lagos  WhatsApp      Electronics   
2          100003 2026-02-26    AB-01  Abuja  WhatsApp      Electronics   
3          100004 2026-06-09    KN-01   Kano    Online      Electronics   
4          100005 2026-05-30    LG-01  Lagos    Online      Electronics   

        product customer_id customer_type  units  unit_price_ngn  \
0  Standing Fan       C1141           New      1           58000   
1    Smartphone       C1083     Returning      5          165000   
2       Earbuds       C1029     Returning      3           32000   
3    Power Bank       C1082     Returning      4           26000   
4    Smartphone       C1180     Returning      1          165000   

   discount_pct  revenue_reported_ngn  cost_per_unit_ngn return_flag sales_rep  
0          0.00                 58000                NaN   

In [5]:
# Validating unit_price and revenue_reported by product
df["expected_revenue_ngn"] = (
    df["units"]
    * df["unit_price_ngn"]
    * (1 - df["discount_pct"])
)

df["revenue_difference_ngn"] = (
    df["revenue_reported_ngn"]
    - df["expected_revenue_ngn"]
)

df["revenue_validation"] = np.where(
    np.isclose(df["revenue_difference_ngn"], 0),
    "PASS",
    "FAIL"
)

In [6]:
print(df[df["revenue_validation"] == "FAIL"])

     transaction_id       date store_id          state   channel  \
1            100002 2026-04-17    LG-01          Lagos  WhatsApp   
15           100016 2026-07-25    LG-02          Lagos    Online   
55           100056 2026-06-18    KN-01           Kano  In-store   
66           100067 2026-07-07    AB-02          Abuja  WhatsApp   
71           100072 2026-06-09    IB-01         Ibadan  In-store   
91           100092 2026-05-11    PH-01  Port Harcourt  In-store   
93           100094 2026-07-20    LG-02          Lagos  In-store   
120          100121 2026-08-14    IB-01         Ibadan  In-store   
162          100163 2026-06-26    KN-01           Kano    Online   
171          100172 2026-03-29    IB-01         Ibadan  In-store   
191          100192 2026-03-07    KN-01           Kano    Online   
232          100233 2026-01-08    IB-01         Ibadan  In-store   
234          100235 2026-05-22    IB-01         Ibadan  WhatsApp   
258          100259 2026-07-23    AB-01         

In [7]:
# Finding B — inconsistent State labels
df["state"] = df["state"].str.strip().str.title()
print(df['state'])

0       Abuja
1       Lagos
2       Abuja
3        Kano
4       Lagos
        ...  
427      Kano
428    Ibadan
429     Lagos
430     Lagos
431    Ibadan
Name: state, Length: 432, dtype: object


In [8]:
df["duplicate_transaction"] = df["transaction_id"].duplicated(
    keep=False
)
print(df["duplicate_transaction"].value_counts())

duplicate_transaction
False    408
True      24
Name: count, dtype: int64


In [11]:
# To see the duplicate transactions
df.loc[
    df["duplicate_transaction"],
    "transaction_id"
].drop_duplicates().sort_values()

60     100061
73     100074
195    100196
196    100197
251    100252
275    100276
303    100304
305    100306
366    100367
385    100386
394    100395
400    100401
Name: transaction_id, dtype: int64

In [12]:
# How many times the transaction occured
df.loc[
    df["duplicate_transaction"],
    "transaction_id"
].value_counts().sort_index()

transaction_id
100061    2
100074    2
100196    2
100197    2
100252    2
100276    2
100304    2
100306    2
100367    2
100386    2
100395    2
100401    2
Name: count, dtype: int64

In [13]:
# Are the duplicated rows actually identical?
duplicate_ids = (
    df.loc[df["duplicate_transaction"], "transaction_id"]
    .drop_duplicates()
)

duplicate_records = (
    df[df["transaction_id"].isin(duplicate_ids)]
    .sort_values("transaction_id")
)

duplicate_records

,transaction_id,date,store_id,state,channel,product_category,product,customer_id,customer_type,units,...,discount_pct,revenue_reported_ngn,cost_per_unit_ngn,return_flag,sales_rep,expected_revenue_ngn,revenue_difference_ngn,revenue_validation,duplicate_transaction,category_validation
60,100061,2026-08-17,LG-02,Lagos,In-store,Electronics,Earbuds,C1039,Returning,1,...,0.15,27200,19000.0,No,Aisha,27200.0,0.0,PASS,True,PASS
430,100061,2026-08-17,LG-02,Lagos,In-store,Electronics,Earbuds,C1039,Returning,1,...,0.15,27200,19000.0,No,Aisha,27200.0,0.0,PASS,True,PASS
73,100074,2026-04-20,IB-01,Ibadan,In-store,Accessories,USB Cable,C1043,New,4,...,0.05,24700,2200.0,No,Chinedu,24700.0,0.0,PASS,True,PASS
428,100074,2026-04-20,IB-01,Ibadan,In-store,Accessories,USB Cable,C1043,New,4,...,0.05,24700,2200.0,No,Chinedu,24700.0,0.0,PASS,True,PASS
195,100196,2026-02-25,LG-02,Lagos,In-store,Electronics,Earbuds,C1147,Returning,4,...,0.10,115200,19000.0,No,Grace,115200.0,0.0,PASS,True,PASS
421,100196,2026-02-25,LG-02,Lagos,In-store,Electronics,Earbuds,C1147,Returning,4,...,0.10,115200,19000.0,No,Grace,115200.0,0.0,PASS,True,PASS
196,100197,2026-08-01,AB-01,Abuja,Online,Accessories,Phone Case,C1152,Returning,1,...,0.25,6750,3500.0,Yes,Kemi,6750.0,0.0,PASS,True,PASS
420,100197,2026-08-01,AB-01,Abuja,Online,Accessories,Phone Case,C1152,Returning,1,...,0.25,6750,3500.0,Yes,Kemi,6750.0,0.0,PASS,True,PASS
425,100252,2026-07-11,KN-01,Kano,In-store,Electronics,Power Bank,C1117,Returning,2,...,0.00,52000,15000.0,No,David,52000.0,0.0,PASS,True,PASS
251,100252,2026-07-11,KN-01,Kano,In-store,Electronics,Power Bank,C1117,Returning,2,...,0.00,52000,15000.0,No,David,52000.0,0.0,PASS,True,PASS


In [15]:
# Compare all column for each pair
duplicate_records.groupby("transaction_id").size()

transaction_id
100061    2
100074    2
100196    2
100197    2
100252    2
100276    2
100304    2
100306    2
100367    2
100386    2
100395    2
100401    2
dtype: int64

In [16]:
duplicate_records.groupby("transaction_id").nunique()

,date,store_id,state,channel,product_category,product,customer_id,customer_type,units,unit_price_ngn,discount_pct,revenue_reported_ngn,cost_per_unit_ngn,return_flag,sales_rep,expected_revenue_ngn,revenue_difference_ngn,revenue_validation,duplicate_transaction,category_validation
transaction_id,,,,,,,,,,,,,,,,,,,,
100061,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100074,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100196,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100197,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100252,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100276,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100304,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100306,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100367,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [17]:
# Remove duplicate transactions
duplicate_ids = (
    df.loc[df["duplicate_transaction"], "transaction_id"]
    .drop_duplicates()
)

duplicate_records = (
    df[df["transaction_id"].isin(duplicate_ids)]
    .sort_values("transaction_id")
)

duplicate_records

,transaction_id,date,store_id,state,channel,product_category,product,customer_id,customer_type,units,...,discount_pct,revenue_reported_ngn,cost_per_unit_ngn,return_flag,sales_rep,expected_revenue_ngn,revenue_difference_ngn,revenue_validation,duplicate_transaction,category_validation
60,100061,2026-08-17,LG-02,Lagos,In-store,Electronics,Earbuds,C1039,Returning,1,...,0.15,27200,19000.0,No,Aisha,27200.0,0.0,PASS,True,PASS
430,100061,2026-08-17,LG-02,Lagos,In-store,Electronics,Earbuds,C1039,Returning,1,...,0.15,27200,19000.0,No,Aisha,27200.0,0.0,PASS,True,PASS
73,100074,2026-04-20,IB-01,Ibadan,In-store,Accessories,USB Cable,C1043,New,4,...,0.05,24700,2200.0,No,Chinedu,24700.0,0.0,PASS,True,PASS
428,100074,2026-04-20,IB-01,Ibadan,In-store,Accessories,USB Cable,C1043,New,4,...,0.05,24700,2200.0,No,Chinedu,24700.0,0.0,PASS,True,PASS
195,100196,2026-02-25,LG-02,Lagos,In-store,Electronics,Earbuds,C1147,Returning,4,...,0.10,115200,19000.0,No,Grace,115200.0,0.0,PASS,True,PASS
421,100196,2026-02-25,LG-02,Lagos,In-store,Electronics,Earbuds,C1147,Returning,4,...,0.10,115200,19000.0,No,Grace,115200.0,0.0,PASS,True,PASS
196,100197,2026-08-01,AB-01,Abuja,Online,Accessories,Phone Case,C1152,Returning,1,...,0.25,6750,3500.0,Yes,Kemi,6750.0,0.0,PASS,True,PASS
420,100197,2026-08-01,AB-01,Abuja,Online,Accessories,Phone Case,C1152,Returning,1,...,0.25,6750,3500.0,Yes,Kemi,6750.0,0.0,PASS,True,PASS
425,100252,2026-07-11,KN-01,Kano,In-store,Electronics,Power Bank,C1117,Returning,2,...,0.00,52000,15000.0,No,David,52000.0,0.0,PASS,True,PASS
251,100252,2026-07-11,KN-01,Kano,In-store,Electronics,Power Bank,C1117,Returning,2,...,0.00,52000,15000.0,No,David,52000.0,0.0,PASS,True,PASS


In [18]:
duplicate_records.groupby("transaction_id").size()

transaction_id
100061    2
100074    2
100196    2
100197    2
100252    2
100276    2
100304    2
100306    2
100367    2
100386    2
100395    2
100401    2
dtype: int64

In [19]:
duplicate_records.groupby("transaction_id").nunique()

,date,store_id,state,channel,product_category,product,customer_id,customer_type,units,unit_price_ngn,discount_pct,revenue_reported_ngn,cost_per_unit_ngn,return_flag,sales_rep,expected_revenue_ngn,revenue_difference_ngn,revenue_validation,duplicate_transaction,category_validation
transaction_id,,,,,,,,,,,,,,,,,,,,
100061,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100074,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100196,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100197,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100252,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100276,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100304,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100306,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
100367,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [9]:
# Product to category validation
product_category = (
    df.groupby("product")["product_category"]
    .first()
    .str.strip()
    .str.title()
    .to_dict()
)

expected_category = df["product"].map(product_category)

df["category_validation"] = np.where(
    df["product_category"].str.strip().str.title()
    == expected_category,
    "PASS",
    "FAIL"
)
print(df["category_validation"])

0      PASS
1      PASS
2      PASS
3      PASS
4      PASS
       ... 
427    PASS
428    PASS
429    PASS
430    PASS
431    PASS
Name: category_validation, Length: 432, dtype: object
